In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import glob
import os
from sklearn.preprocessing import LabelEncoder

# Constants
DATA_DIR = "train"
MAX_INPUT_FRAMES = 50
MAX_OUTPUT_FRAMES = 25
FIELD_X, FIELD_Y = 120.0, 53.3
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

print(f"Using device: {device}")


Using device: mps


In [2]:
# Load and preprocess training data
def load_tracking_data(data_dir):
    input_files = sorted(glob.glob(os.path.join(data_dir, "input_2023_w*.csv")))
    output_files = sorted(glob.glob(os.path.join(data_dir, "output_2023_w*.csv")))
    df_input = pd.concat((pd.read_csv(f) for f in input_files), ignore_index=True)
    df_output = pd.concat((pd.read_csv(f) for f in output_files), ignore_index=True)
    return df_input, df_output

df_input, df_output = load_tracking_data(DATA_DIR)
print(f"Loaded {len(df_input)} input records, {len(df_output)} output records")


Loaded 4880579 input records, 562936 output records


In [ ]:
# Preprocess data
drop_cols = ["player_name", "player_height", "player_weight", "player_birth_date"]
df_input.drop(columns=drop_cols, inplace=True, errors="ignore")

# Store original play_direction for flipping
play_directions = df_input[['game_id', 'play_id', 'play_direction']].drop_duplicates()

# Encode categorical variables
cat_cols = ["player_position", "player_role", "player_side", "play_direction"]
encoders = {col: LabelEncoder().fit(df_input[col].astype(str)) for col in cat_cols}

for col, le in encoders.items():
    df_input[col] = le.transform(df_input[col].astype(str))

# Flip coordinates based on play direction (standardize to left-to-right)
def flip_coordinates(df, flip_cols=['x', 'ball_land_x']):
    """Flip coordinates for right-to-left plays"""
    df_flipped = df.copy()
    right_to_left_mask = df['play_direction'] == 1  # Assuming 1 = right-to-left after encoding
    
    for col in flip_cols:
        if col in df_flipped.columns:
            df_flipped.loc[right_to_left_mask, col] = FIELD_X - df_flipped.loc[right_to_left_mask, col]
    
    return df_flipped

# Apply coordinate flipping to input data
df_input = flip_coordinates(df_input, ['x', 'ball_land_x'])

# Apply coordinate flipping to output data using play_directions mapping
df_output_merged = df_output.merge(play_directions, on=['game_id', 'play_id'], how='left')
df_output_merged['play_direction'] = encoders['play_direction'].transform(df_output_merged['play_direction'].astype(str))
df_output = flip_coordinates(df_output_merged, ['x'])

# Normalize features
df_input["x_norm"] = df_input["x"] / FIELD_X
df_input["y_norm"] = df_input["y"] / FIELD_Y
df_input["s_norm"] = df_input["s"] / df_input["s"].max()
df_input["a_norm"] = df_input["a"] / df_input["a"].max()
df_output["x"] = df_output["x"] / FIELD_X
df_output["y"] = df_output["y"] / FIELD_Y

# Velocity components
df_input["dir_rad"] = np.deg2rad(df_input["dir"])
df_input["vx"] = df_input["s_norm"] * np.cos(df_input["dir_rad"])
df_input["vy"] = df_input["s_norm"] * np.sin(df_input["dir_rad"])

# Ball-relative features
df_input["x_rel_to_ball"] = (df_input["x"] - df_input["ball_land_x"]) / FIELD_X
df_input["y_rel_to_ball"] = (df_input["y"] - df_input["ball_land_y"]) / FIELD_Y
df_input["dist_to_ball"] = np.sqrt(df_input["x_rel_to_ball"]**2 + df_input["y_rel_to_ball"]**2)

feature_cols = ["x_norm", "y_norm", "vx", "vy", "a_norm", "player_position", 
                "player_role", "player_side", "play_direction", "x_rel_to_ball", 
                "y_rel_to_ball", "dist_to_ball"]

print("Data preprocessing completed with coordinate flipping")


KeyError: 'play_direction'

In [ ]:
# Prepare training data
plays = df_input.groupby(["game_id", "play_id"])
X_data, Y_data, masks = [], [], []

for (game_id, play_id), play_df in plays:
    # Input frames
    frames_in = play_df.sort_values("frame_id")
    grouped_frames = []
    for frame_id, frame_df in frames_in.groupby("frame_id"):
        mat = frame_df[feature_cols].to_numpy()
        if mat.shape[0] < 22:
            mat = np.pad(mat, ((0, 22 - mat.shape[0]), (0, 0)), mode="constant")
        grouped_frames.append(mat)
    
    if not grouped_frames:
        continue
        
    X_seq = np.stack(grouped_frames)
    if X_seq.shape[0] < MAX_INPUT_FRAMES:
        pad_len = MAX_INPUT_FRAMES - X_seq.shape[0]
        last_frame = X_seq[-1:]
        pad_frames = np.repeat(last_frame, pad_len, axis=0)
        X_seq = np.concatenate([X_seq, pad_frames], axis=0)
    else:
        X_seq = X_seq[:MAX_INPUT_FRAMES]

    # Output frames
    play_out = df_output[(df_output["game_id"] == game_id) & (df_output["play_id"] == play_id)]
    frames_out = []
    for frame_id, frame_df in play_out.groupby("frame_id"):
        mat = frame_df[["x", "y"]].to_numpy()
        if mat.shape[0] < 22:
            mat = np.pad(mat, ((0, 22 - mat.shape[0]), (0, 0)), mode="constant")
        frames_out.append(mat)
    
    if not frames_out:
        continue
        
    Y_seq = np.stack(frames_out)
    if Y_seq.shape[0] < MAX_OUTPUT_FRAMES:
        pad_len = MAX_OUTPUT_FRAMES - Y_seq.shape[0]
        last_frame = Y_seq[-1:]
        pad_frames = np.repeat(last_frame, pad_len, axis=0)
        Y_seq = np.concatenate([Y_seq, pad_frames], axis=0)
    else:
        Y_seq = Y_seq[:MAX_OUTPUT_FRAMES]

    # Mask for predicted players
    mask = frames_in.groupby("nfl_id")["player_to_predict"].max().values
    if len(mask) < 22:
        mask = np.pad(mask, (0, 22 - len(mask)), mode="constant")

    X_data.append(X_seq)
    Y_data.append(Y_seq)
    masks.append(mask)

X_data = np.array(X_data)
Y_data = np.array(Y_data)
masks = np.array(masks)

print(f"Training data shapes: X={X_data.shape}, Y={Y_data.shape}, masks={masks.shape}")


In [ ]:
# Improved Seq-to-Seq Graph Neural Network
class FrameGraphModule(nn.Module):
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.intra_team_fc = nn.Linear(feat_dim, hidden_dim)
        self.bipartite_fc = nn.Linear(feat_dim, hidden_dim)
        self.to_landing_fc = nn.Linear(feat_dim, hidden_dim)
        self.out_fc = nn.Linear(hidden_dim * 3, feat_dim)

    def forward(self, x, adj_intra, adj_bipartite, adj_to_landing):
        intra_msg = torch.matmul(adj_intra, self.intra_team_fc(x))
        bip_msg = torch.matmul(adj_bipartite, self.bipartite_fc(x))
        landing_msg = torch.matmul(adj_to_landing, self.to_landing_fc(x))
        combined = torch.cat([intra_msg, bip_msg, landing_msg], dim=-1)
        return self.out_fc(combined) + x

class Seq2SeqGraphTransformer(nn.Module):
    def __init__(self, feat_dim, hidden_dim, n_heads=4, n_layers=2, seq_out=25):
        super().__init__()
        self.frame_graph = FrameGraphModule(feat_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=22*feat_dim, nhead=n_heads, batch_first=True)
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.seq_out = seq_out
        self.feat_dim = feat_dim
        self.output_proj = nn.Linear(22*feat_dim, 22*2)

    def forward(self, x, adj_intra_seq, adj_bipartite_seq, adj_to_landing_seq):
        batch, seq_in, num_players, feat_dim = x.shape
        
        # Vectorized frame-wise graph updates - process all frames at once
        x_bt = x.view(batch * seq_in, num_players, feat_dim)
        adj_intra_bt = adj_intra_seq.view(batch * seq_in, num_players, num_players)
        adj_bipartite_bt = adj_bipartite_seq.view(batch * seq_in, num_players, num_players)
        adj_to_landing_bt = adj_to_landing_seq.view(batch * seq_in, num_players, num_players)

        # Graph operations on all frames simultaneously
        intra_msg = torch.matmul(adj_intra_bt, self.frame_graph.intra_team_fc(x_bt))
        bip_msg = torch.matmul(adj_bipartite_bt, self.frame_graph.bipartite_fc(x_bt))
        landing_msg = torch.matmul(adj_to_landing_bt, self.frame_graph.to_landing_fc(x_bt))

        # Concatenate messages and residual
        combined = torch.cat([intra_msg, bip_msg, landing_msg], dim=-1)
        x_bt_out = self.frame_graph.out_fc(combined) + x_bt

        # Reshape back to sequence format
        x_seq = x_bt_out.view(batch, seq_in, num_players * feat_dim)

        # Temporal transformer
        x_seq = self.temporal_encoder(x_seq)

        # Improved autoregressive prediction with state updates
        preds = []
        hidden_state = x_seq[:, -1, :]  # Start with last input frame
        
        for t in range(self.seq_out):
            # Predict next frame
            out = self.output_proj(hidden_state)  # (batch, 22*2)
            pred_frame = out.view(batch, num_players, 2)
            preds.append(pred_frame)
            
            # Update hidden state for next prediction
            pred_features = pred_frame.view(batch, num_players * 2)
            if not hasattr(self, 'pred_to_feat'):
                self.pred_to_feat = nn.Linear(num_players * 2, num_players * feat_dim).to(hidden_state.device)
            
            # Update hidden state with predicted features
            hidden_state = hidden_state + 0.1 * self.pred_to_feat(pred_features)  # Residual update
            
        return torch.stack(preds, dim=1)

# Create model
feat_dim = X_data.shape[-1]
hidden_dim = 64
graph_model = Seq2SeqGraphTransformer(feat_dim=feat_dim, hidden_dim=hidden_dim, seq_out=MAX_OUTPUT_FRAMES)
graph_model.to(device)
optimizer = torch.optim.Adam(graph_model.parameters(), lr=5e-4)

print(f"Graph model created with {sum(p.numel() for p in graph_model.parameters()):,} parameters")


In [ ]:
# Compute adjacency matrices
def compute_adjacency_seq(X_data, df_input):
    batch_size, seq_in, num_players, feat_dim = X_data.shape
    adj_intra_seq = np.zeros((batch_size, seq_in, num_players, num_players))
    adj_bipartite_seq = np.zeros_like(adj_intra_seq)
    adj_to_landing_seq = np.zeros_like(adj_intra_seq)

    plays = df_input.groupby(["game_id", "play_id"])
    
    for batch_idx, ((game_id, play_id), play_df) in enumerate(plays):
        if batch_idx >= batch_size:
            break
        
        frames_in = play_df.sort_values("frame_id")
        
        for frame_idx, (frame_id, frame_df) in enumerate(frames_in.groupby("frame_id")):
            if frame_idx >= seq_in:
                break
            
            player_sides = frame_df['player_side'].values
            present = min(len(player_sides), num_players)
            
            for i in range(present):
                for j in range(present):
                    if i == j:
                        continue
                    if player_sides[i] == player_sides[j]:
                        adj_intra_seq[batch_idx, frame_idx, i, j] = 1
                    else:
                        adj_bipartite_seq[batch_idx, frame_idx, i, j] = 1
            
            for i in range(present):
                for j in range(present):
                    if i != j:
                        adj_to_landing_seq[batch_idx, frame_idx, i, j] = 1

    return (
        torch.tensor(adj_intra_seq, dtype=torch.float32),
        torch.tensor(adj_bipartite_seq, dtype=torch.float32),
        torch.tensor(adj_to_landing_seq, dtype=torch.float32),
    )

# Precompute adjacencies
print("Computing adjacency matrices...")
adj_intra_all, adj_bipartite_all, adj_to_landing_all = compute_adjacency_seq(X_data, df_input)
adj_intra_all = adj_intra_all.to(device)
adj_bipartite_all = adj_bipartite_all.to(device)
adj_to_landing_all = adj_to_landing_all.to(device)
print("Adjacency matrices computed")


In [ ]:
# Training
def masked_rmse(pred, target, mask):
    mask = mask.unsqueeze(1).expand(-1, pred.shape[1], -1).unsqueeze(-1)
    diff = (pred - target) ** 2
    masked_diff = diff * mask.float()
    mse = masked_diff.sum() / mask.sum().float()
    return torch.sqrt(mse)

print("Training improved graph model...")
for epoch in range(5):
    graph_model.train()
    total_loss = 0
    batch_size = 16
    
    for i in range(0, len(X_data), batch_size):
        batch_end = min(i + batch_size, len(X_data))
        X_batch = torch.tensor(X_data[i:batch_end], dtype=torch.float32, device=device)
        Y_batch = torch.tensor(Y_data[i:batch_end], dtype=torch.float32, device=device)
        mask_batch = torch.tensor(masks[i:batch_end], dtype=torch.bool, device=device)
        
        adj_intra = adj_intra_all[i:batch_end]
        adj_bipartite = adj_bipartite_all[i:batch_end]
        adj_to_landing = adj_to_landing_all[i:batch_end]
        
        pred = graph_model(X_batch, adj_intra, adj_bipartite, adj_to_landing)
        loss = masked_rmse(pred, Y_batch, mask_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(graph_model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/(len(X_data)//batch_size):.5f}")

print("Training completed!")


In [ ]:
# Load and preprocess test data
test_input = pd.read_csv("test_input.csv")
test_submission_format = pd.read_csv("sample_submission.csv")

print(f"Test input shape: {test_input.shape}")
print(f"Test submission format shape: {test_submission_format.shape}")

# Process test data
test_input_processed = test_input.copy()
test_input_processed.drop(columns=drop_cols, inplace=True, errors="ignore")

# Apply same preprocessing
for col in cat_cols:
    if col in test_input_processed.columns:
        test_input_processed[col] = encoders[col].transform(test_input_processed[col].astype(str))

# Apply coordinate flipping to test data (same as training)
test_input_processed = flip_coordinates(test_input_processed, ['x', 'ball_land_x'])

test_input_processed["x_norm"] = test_input_processed["x"] / FIELD_X
test_input_processed["y_norm"] = test_input_processed["y"] / FIELD_Y
test_input_processed["s_norm"] = test_input_processed["s"] / test_input_processed["s"].max()
test_input_processed["a_norm"] = test_input_processed["a"] / test_input_processed["a"].max()
test_input_processed["dir_rad"] = np.deg2rad(test_input_processed["dir"])
test_input_processed["vx"] = test_input_processed["s_norm"] * np.cos(test_input_processed["dir_rad"])
test_input_processed["vy"] = test_input_processed["s_norm"] * np.sin(test_input_processed["dir_rad"])
test_input_processed["x_rel_to_ball"] = (test_input_processed["x"] - test_input_processed["ball_land_x"]) / FIELD_X
test_input_processed["y_rel_to_ball"] = (test_input_processed["y"] - test_input_processed["ball_land_y"]) / FIELD_Y
test_input_processed["dist_to_ball"] = np.sqrt(test_input_processed["x_rel_to_ball"]**2 + test_input_processed["y_rel_to_ball"]**2)

print("Test data preprocessing completed with coordinate flipping")


In [ ]:
# Generate predictions
test_plays = test_input_processed.groupby(["game_id", "play_id"])
submission_data = []

graph_model.eval()
with torch.no_grad():
    for (game_id, play_id), play_df in test_plays:
        frames_in = play_df.sort_values("frame_id")
        grouped_frames = []
        
        for frame_id, frame_df in frames_in.groupby("frame_id"):
            mat = frame_df[feature_cols].to_numpy()
            if mat.shape[0] < 22:
                mat = np.pad(mat, ((0, 22 - mat.shape[0]), (0, 0)), mode="constant")
            grouped_frames.append(mat)
        
        if not grouped_frames:
            continue
            
        X_seq = np.stack(grouped_frames)
        if X_seq.shape[0] < MAX_INPUT_FRAMES:
            pad_len = MAX_INPUT_FRAMES - X_seq.shape[0]
            last_frame = X_seq[-1:]
            pad_frames = np.repeat(last_frame, pad_len, axis=0)
            X_seq = np.concatenate([X_seq, pad_frames], axis=0)
        else:
            X_seq = X_seq[:MAX_INPUT_FRAMES]
        
        # Compute adjacency matrices for test play
        adj_intra, adj_bipartite, adj_to_landing = compute_adjacency_seq(
            X_seq.reshape(1, X_seq.shape[0], X_seq.shape[1], X_seq.shape[2]), 
            play_df
        )
        
        X_tensor = torch.tensor(X_seq, dtype=torch.float32).unsqueeze(0).to(device)
        adj_intra = adj_intra.to(device)
        adj_bipartite = adj_bipartite.to(device)
        adj_to_landing = adj_to_landing.to(device)
        
        pred = graph_model(X_tensor, adj_intra, adj_bipartite, adj_to_landing).cpu().numpy()[0]
        
        # Get play direction for this play to determine if we need to flip back
        play_direction_original = play_df['play_direction'].iloc[0]
        is_right_to_left = play_direction_original == 1  # Assuming 1 = right-to-left after encoding
        
        # Map predictions to submission format
        play_test_data = test_submission_format[
            test_submission_format['id'].str.startswith(f"{game_id}_{play_id}")
        ]
        
        unique_nfl_ids = play_df['nfl_id'].unique()
        nfl_id_to_idx = {nfl_id: idx for idx, nfl_id in enumerate(unique_nfl_ids)}
        
        for _, row in play_test_data.iterrows():
            id_parts = row['id'].split('_')
            nfl_id = int(id_parts[2])
            frame_id = int(id_parts[3])
            
            if nfl_id in nfl_id_to_idx and frame_id <= 25:
                player_idx = nfl_id_to_idx[nfl_id]
                pred_x = pred[frame_id - 1, player_idx, 0] * FIELD_X
                pred_y = pred[frame_id - 1, player_idx, 1] * FIELD_Y
                
                # Flip back if this was a right-to-left play
                if is_right_to_left:
                    pred_x = FIELD_X - pred_x
                
                submission_data.append({'id': row['id'], 'x': pred_x, 'y': pred_y})
            else:
                submission_data.append({'id': row['id'], 'x': 0.0, 'y': 0.0})

# Save submission
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('submission_seq2seqgnn.csv', index=False)

print(f"Submission saved: {len(submission_df)} predictions")
print("Sample predictions:")
print(submission_df.head())
print("\nSubmission file: submission_seq2seqgnn.csv")
